# RnD Data Lake — how to access data

This notebook shows how to read data from the **rnd-data-lake** (repo: `/cache/fast_data_nas71/janhavi/rnd-data-lake`).
It is *not* about the swift training format — it's here to build intuition for how the data lake is laid out and how to pull
CT volumes + reports/metadata out of it.

## The one big idea: metadata and pixels are two separate systems

| Layer | Where | Holds | How you touch it |
|---|---|---|---|
| **Labels store** | `src/rnd_data_lake/labels/STORE/` (parquet, DVC-tracked) | DICOM Patient/Study/Series/Instance graph + reports + findings. **No pixels.** | DuckDB SQL via `get_connection()` |
| **Blob cache** | `/mnt/cache/blobs/` (content-addressed, S3-backed) | The actual **CT pixel data** as `.safetensors`, plus `.png` previews | `get_cached_path(s3_uri)` |

The bridge between them is the **`SeriesInstanceUID`**: you query the labels store to decide *which* scans you want,
then use that UID to fetch the *pixels* from the cache.

> **Environment:** run this with a Python that has `rnd_data_lake` installed (e.g. the rnd-data-lake `uv`/conda env),
> plus `duckdb`, `safetensors`, `numpy`, `matplotlib`, `pillow`. Pixel access also needs the **`nas` boto profile**
> configured (see the repo root `README.md`).
>
> **Important — pin the store path.** `get_connection()` with no args resolves the store *relative to wherever
> `rnd_data_lake` is installed in your kernel*. If that install is a non-`dvc pull`-hydrated copy, the views come back
> empty (you'll see `BinderException: column "CommonMetadata.Modality" not found ... Candidate bindings: "node_id"`).
> The cell below passes an explicit `StoreLayout` pointing at the hydrated store to avoid that.


In [1]:
from rnd_data_lake.labels.db.connection import get_connection
from rnd_data_lake.labels.utils.layout import StoreLayout
from rnd_data_lake.cache import get_cached_path
import numpy as np
import pandas as pd

# Pin to the hydrated (dvc-pulled) store rather than relying on the install-relative default.
STORE_PATH = "/cache/fast_data_nas71/janhavi/rnd-data-lake/src/rnd_data_lake/labels/STORE"
con = get_connection(layout=StoreLayout(STORE_PATH))   # all node/edge views registered

# sanity check: a hydrated nodes_series has ~1000 DICOM columns, not just node_id
ncols = con.execute("PRAGMA table_info('nodes_series')").fetchdf().shape[0]
assert ncols > 10, f"nodes_series has only {ncols} cols -> store not hydrated; check STORE_PATH / dvc pull"
print(f"connected OK; nodes_series has {ncols} columns")
con.execute("SELECT COUNT(*) FROM nodes_series").fetchone()

connected OK; nodes_series has 1028 columns


(4091360,)

---
# Section 1 — Reports & metadata (the labels store)

Everything here is plain DuckDB SQL over pre-registered **views**. Key ones:

| View | Grain | Notes |
|---|---|---|
| `nodes_series` | one row / series | DICOM metadata **+ denormalized** `PatientID`, `StudyInstanceUID`, `StudyDate`, demographics |
| `nodes_series_computed` | one row / series | `nodes_series` + geometry (`is_axial`, `image_plane`, `spacing_*`) |
| `nodes_instance` | one row / SOP instance | flat DICOM columns; carries `SeriesInstanceUID` |
| `nodes_report` | one row / report | only column is `text` (free-text radiology report) |
| `nodes_source` | one row / provider | only column is `name` (e.g. `segmed`, `nlst`, `ct_rate`) |
| `edges` | typed directed graph | hierarchy + annotations + provenance |

Rules of thumb (baked into the store's query skill):
- Read patient/study facts straight off `nodes_series` — **don't** join `edges` for them.
- `edges` is the expensive view: only touch it for real graph links (reports, findings, source), and **always** add `WHERE source_type='...'`.
- Prefer the `"CommonMetadata.X"` column over its top-level twin; dotted names must be double-quoted.
- Views already apply latest-wins + drop deletes — never add your own `created_at`/`ROW_NUMBER` dedup.


### What each view *is*

The store is a **graph** mirroring the DICOM hierarchy: a **Patient** has **Studies** (imaging visits), a study has
**Series** (individual scans/reconstructions), a series has **Instances** (single slices / SOP objects). Plus
provenance nodes: **Report** (the radiology text) and **Source** (which provider supplied the data). Each node type
gets one view; `edges` is the graph that wires them together.

- **`nodes_patient`** — one row per patient. A bare **stub**: essentially just `node_id` + `PatientID`. No demographics
  live here (they're denormalized onto the series row for query speed).
- **`nodes_study`** — one row per study (an imaging visit / accession). Also a **stub**: `node_id` + `StudyInstanceUID`.
- **`nodes_series`** — one row per **series** = one scan/reconstruction. **This is the workhorse table** (~1000 DICOM
  columns). It carries the series' own DICOM metadata *and* denormalizes its ancestors' identifiers/demographics
  (`PatientID`, `StudyInstanceUID`, `StudyDate`, age, sex, `Modality`, `BodyPartExamined`, `NumInstances`, ...). Because
  the ancestor IDs are copied down here, you rarely need `edges` to filter by patient or study.
- **`nodes_instance`** — one row per **SOP instance** (typically a single slice). Flat DICOM columns; carries
  `SeriesInstanceUID` (its parent) but **not** study/patient. Big table (123M+ rows). You usually don't need it -
  per-series slice count is already on `nodes_series.NumInstances`.
- **`nodes_series_computed`** — `nodes_series` **plus derived geometry** columns computed by UDFs from the raw DICOM
  tags: `is_axial`/`is_coronal`/`is_sagittal`/`is_oblique`, `image_plane`, and voxel spacing
  (`spacing_x_patient`/`y`/`z`, ...). Use this whenever you need plane or spacing; never recompute geometry by hand.
- **`nodes_report`** — one row per clinical **report**. The only data column is `text` (free-text radiology report,
  anonymized). Content-addressed (`node_id = report_<sha256(text)>`), so identical text collapses to one node.
- **`nodes_source`** — one row per originating **provider** (`name`, e.g. `segmed`, `nlst`, `ct_rate`). One node per
  provider, shared across all its studies.
- **`edges`** — the typed, directed graph linking everything. Each row is `(source_id, source_type, edge_type,
  target_id)`. Three families:
  - **hierarchy** (child -> parent): `instance --instance_of--> series --series_of--> study --study_of--> patient`
  - **provenance**: `report --report_of--> study`, and `study --from_source--> source`
  - **annotations** (when present): `finding/bbox/mask --on_series|on_instance|...--> node`
  It's the most expensive view (it unions every edge delta at read time), so use it only for genuine graph links and
  always prune with `WHERE source_type = '<type>'`.

Every node view also exposes framework columns: `node_id`, `node_type`, `created_at`, `created_by`, `operation`,
`content_hash`. Node-id formats: `patient_<PatientID>`, `study_<StudyInstanceUID>`, `series_<SeriesInstanceUID>`,
`instance_<SOPInstanceUID>`, `report_<sha256(text)>`, `source_<sha256(name)>`.


### 1a. Discover the schema

In [ ]:
# list all registered views
print([r[0] for r in con.execute("SHOW TABLES").fetchall()])

# nodes_series has ~190 DICOM columns — peek at a few
cols = con.execute("PRAGMA table_info('nodes_series')").fetchdf()
print(cols.shape[0], "columns on nodes_series; e.g.:")
cols[['name','type']].head(20)

### 1b. Basic counts / modality filter

In [ ]:
con.execute('''
  SELECT "CommonMetadata.Modality" AS modality, COUNT(*) AS n
  FROM nodes_series
  GROUP BY 1 ORDER BY n DESC
  LIMIT 10
''').fetchdf()

### 1c. Query CT series with useful metadata

In [ ]:
df = con.execute('''
  SELECT
    SeriesInstanceUID,
    StudyInstanceUID,
    PatientID,
    StudyDate,
    "CommonMetadata.Modality"        AS modality,
    "CommonMetadata.BodyPartExamined" AS body_part,
    TRY_CAST(NumInstances AS BIGINT)   AS num_slices
  FROM nodes_series
  WHERE "CommonMetadata.Modality" = 'CT'
    AND TRY_CAST(NumInstances AS BIGINT) BETWEEN 40 AND 500
  LIMIT 10
''').fetchdf()
df

### 1d. Geometry — restrict to axial CT & read voxel spacing

Use `nodes_series_computed` (never hand-roll geometry from raw DICOM tags). This is how you'd keep only axial
reconstructions for training and know the real slice spacing.

In [ ]:
con.execute('''
  SELECT
    SeriesInstanceUID,
    image_plane,
    is_axial,
    ROUND(spacing_x_patient, 3) AS sx,
    ROUND(spacing_y_patient, 3) AS sy,
    ROUND(spacing_z_patient, 3) AS sz
  FROM nodes_series_computed
  WHERE "CommonMetadata.Modality" = 'CT'
    AND is_axial
  LIMIT 10
''').fetchdf()

### 1e. Reports

A report is a `nodes_report` row (only column: `text`) attached to a **study** via a `report_of` edge:

```
report --report_of--> study        (source_type='report', edge_type='report_of')
```

Node-id conventions: a report is `report_<sha256(text)>`, a study is `study_<StudyInstanceUID>`.
Reports are anonymized (names/IDs are hashed in the text).

In [ ]:
print("total reports:", con.execute("SELECT COUNT(*) FROM nodes_report").fetchone()[0])

# one example report
print(con.execute("SELECT text FROM nodes_report LIMIT 1").fetchone()[0][:600])

### 1f. Join CT series → its study's report

This is the core join for building CT report-generation data: a CT **series** carries `StudyInstanceUID`, and the
**report** hangs off that study. (Reports are per-study, so every CT series in a study shares its report.)

In [ ]:
ct_reports = con.execute('''
  WITH rep AS (                                   -- report -> study, via the report_of edge
    SELECT e.target_id AS study_nodeid, r.text AS report
    FROM edges e
    JOIN nodes_report r ON r.node_id = e.source_id
    WHERE e.source_type = 'report' AND e.edge_type = 'report_of'
  )
  SELECT
    s.SeriesInstanceUID,
    s.StudyInstanceUID,
    rep.report
  FROM nodes_series s
  JOIN rep ON rep.study_nodeid = 'study_' || s.StudyInstanceUID
  WHERE s."CommonMetadata.Modality" = 'CT'
  LIMIT 5
''').fetchdf()

print("rows:", len(ct_reports))
print(ct_reports[['SeriesInstanceUID','StudyInstanceUID']])
print("\n--- example report ---\n", ct_reports.iloc[0]['report'][:800])

### 1g. Provenance — which source did a scan come from?

Studies attach to a `Source` (e.g. `segmed`, `nlst`, `ct_rate`) via a `from_source` edge. Useful because different
cohorts have different assets (see the caveat at the end: NLST has volumes but no reports; Segmed has reports).

In [ ]:
con.execute('''
  SELECT so.name AS source, COUNT(DISTINCT e.source_id) AS n_studies
  FROM edges e
  JOIN nodes_source so ON so.node_id = e.target_id
  WHERE e.source_type = 'study' AND e.edge_type = 'from_source'
  GROUP BY 1 ORDER BY n_studies DESC
''').fetchdf()

---
# Section 2 — Volumes (the pixel blobs)

Pixels live in S3 (Ceph RGW), addressed **deterministically by `SeriesInstanceUID`**:

```
s3://rnd-data-lake/safetensors/<UID>.safetensors   # the CT volume
s3://rnd-data-lake/png/<UID>.png                    # a rendered preview
```

You never touch S3 or `/mnt/cache` directly — you call **`get_cached_path(s3_uri)`**, which returns a local
`pathlib.Path`, downloading the blob on a cache miss and serving it from disk on every hit after that.

> Do **not** construct `/mnt/cache/blobs/...` paths yourself and never write into that tree — it is
> content-addressed and anything the cache didn't put there gets reclaimed.

### 2a. Resolve a UID to a local file

In [ ]:
# A CT series that actually has a volume in the bucket (NLST cohort).
# You'd normally get UIDs from a Section-1 query; hard-coded here so the notebook runs standalone.
uid = "1.2.840.113654.2.55.100008301831234956048352598773808539424"

s3_uri = f"s3://rnd-data-lake/safetensors/{uid}.safetensors"
local_path = get_cached_path(s3_uri)     # downloads on first call, then cached
print(local_path)
print("size (MB):", round(local_path.stat().st_size / 1e6, 1))

### 2b. What's inside a `.safetensors` volume

Each CT blob holds exactly **two tensors**:

| tensor | dtype | shape | meaning |
|---|---|---|---|
| `image` | int16 | `[num_slices, H, W]` | the CT volume, in **raw Hounsfield Units** (no windowing applied) |
| `instance_number` | uint16 | `[num_slices]` | the DICOM InstanceNumber of each slice |

H/W and slice count vary series-to-series, so any downstream pipeline must resample.

In [ ]:
from safetensors.numpy import load_file

d = load_file(local_path)
for k, v in d.items():
    print(f"{k:16s} dtype={str(v.dtype):8s} shape={v.shape}")

vol = d["image"]           # (Z, H, W) int16 HU
inst = d["instance_number"] # (Z,) uint16
print("\nHU stats: min", vol.min(), "max", vol.max(), "mean", round(float(vol.mean()), 1))
print("(air ~= -1000 HU, water ~= 0, dense bone ~= +1000)")

### 2c. Slice ordering — `instance_number`

The slices in `image` are **not guaranteed** to be in anatomical order — `instance_number` gives the DICOM
InstanceNumber of each slice. Sort by it to get a canonical top→bottom ordering. (In swift we deliberately do
**not** force this order in the loader — flipping/shuffling slice order can be used as a 3D augmentation.)

In [ ]:
print("instance_number, first 10:", inst[:10].tolist())
print("already ascending? ", bool(np.all(np.diff(inst.astype(int)) >= 0)))

order = np.argsort(inst)          # canonical ordering
vol_ordered = vol[order]
print("ordered instance_number, first 10:", inst[order][:10].tolist())

### 2d. Visualize a middle slice (with HU windowing)

In [ ]:
import matplotlib.pyplot as plt

def apply_window(img_hu, level, width):
    lo, hi = level - width / 2, level + width / 2
    return np.clip((img_hu.astype(np.float32) - lo) / (hi - lo), 0, 1)

mid = vol_ordered[len(vol_ordered) // 2].astype(np.float32)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(apply_window(mid, level=-600, width=1500), cmap="gray")   # lung window
ax[0].set_title("lung window (WL -600 / WW 1500)"); ax[0].axis("off")
ax[1].imshow(apply_window(mid, level=40, width=400), cmap="gray")      # mediastinal / soft-tissue
ax[1].set_title("soft-tissue window (WL 40 / WW 400)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

### 2e. The PNG preview (cheap thumbnail, no HU math)

A rendered preview *may* exist at `s3://rnd-data-lake/png/<UID>.png`. Not every series has one (e.g. NLST volumes
often don't), so a missing object raises `S3NotFound` — handle it rather than assume it's there.

In [ ]:
from PIL import Image
from rnd_data_lake.cache.errors import S3NotFound

try:
    png_path = get_cached_path(f"s3://rnd-data-lake/png/{uid}.png")
    img = Image.open(png_path)
    print("preview size:", img.size)
    plt.figure(figsize=(4, 4)); plt.imshow(img, cmap="gray"); plt.axis("off"); plt.show()
except S3NotFound:
    print("no PNG preview for this UID — previews aren't uploaded for every series; "
          "use the safetensors volume above instead.")

---
# Section 3 — Putting it together

A single helper that, given a `SeriesInstanceUID`, hands back the ordered HU volume — the pattern any dataset
builder or dataloader would wrap.

In [ ]:
def load_volume_hu(uid, order_by_instance=True):
    """UID -> (Z, H, W) int16 HU volume via the content cache."""
    path = get_cached_path(f"s3://rnd-data-lake/safetensors/{uid}.safetensors")
    d = load_file(path)
    vol, inst = d["image"], d["instance_number"]
    if order_by_instance:
        vol = vol[np.argsort(inst)]
    return vol

v = load_volume_hu(uid)
print(v.shape, v.dtype, "HU", v.min(), "..", v.max())

## Volumes vs reports — the overlap is real (verify it)

Two cohorts live here: **reports** are mostly **Segmed** CT; **volumes** (`s3://rnd-data-lake/safetensors/`) span
several sources. They are **not** disjoint: of the CT series that have a report and are real volumes (>= 40 slices),
a large fraction actually have a safetensors blob (~0.77M at time of writing).

Building the `s3://.../<UID>.safetensors` URI does **not** prove the file exists — always verify against the bucket
before training (Section 4 does this). For text-only LLM tag-extraction you don't need volumes at all: every
report-bearing CT study is usable.

---
# Section 4 — Build a trainable set: CT volumes that have reports

1. `segmed_df`: every CT series that **has a report** and is a **real volume** (>= 40 slices, drops scouts/localizers).
2. Then **verify** which of those series actually have a `.safetensors` blob in the bucket (a constructed `volume_uri`
   does not guarantee the file exists) and keep only those as `trainable_df`.

In [ ]:
segmed_df = con.execute("""
WITH rep AS (                                                    -- report text, keyed by study
    SELECT DISTINCT substr(e.target_id, 7) AS StudyInstanceUID,  -- strip 'study_' prefix
           r.text AS report
    FROM edges e
    JOIN nodes_report r ON r.node_id = e.source_id
    WHERE e.source_type = 'report' AND e.edge_type = 'report_of'
)
SELECT
    s.SeriesInstanceUID,
    s.StudyInstanceUID,
    s.PatientID,
    TRY_CAST(s.NumInstances AS BIGINT) AS num_slices,
    's3://rnd-data-lake/safetensors/' || s.SeriesInstanceUID || '.safetensors' AS volume_uri,
    rep.report
FROM nodes_series s
JOIN rep ON rep.StudyInstanceUID = s.StudyInstanceUID
WHERE s."CommonMetadata.Modality" = 'CT'
  AND TRY_CAST(s.NumInstances AS BIGINT) >= 40                   -- real volumes only
""").fetchdf()

# NOTE: ~2.7M rows; report text repeats once per series, so this DataFrame is several GB.
# For one volume per study, add: QUALIFY ROW_NUMBER() OVER (PARTITION BY s.StudyInstanceUID
#                                    ORDER BY TRY_CAST(s.NumInstances AS BIGINT) DESC) = 1
print("CT series with a report and >=40 slices:", len(segmed_df))
segmed_df.head(3)

In [ ]:
# A constructed volume_uri does NOT prove the blob exists. List the safetensors bucket once
# (~0.97M keys, ~90s over RGW) and keep only the series whose file is actually present.
import boto3, botocore

s3 = boto3.Session(profile_name="nas").client(
    "s3", endpoint_url="http://192.168.1.60:7480",
    config=botocore.config.Config(s3={"addressing_style": "path"}))

have = set()
for page in s3.get_paginator("list_objects_v2").paginate(Bucket="rnd-data-lake", Prefix="safetensors/"):
    have |= {o["Key"].split("/")[-1][:-len(".safetensors")] for o in page.get("Contents", [])}
print("safetensors objects in bucket:", len(have))

segmed_df["has_volume"] = segmed_df["SeriesInstanceUID"].isin(have)
print("series with a report that ACTUALLY have a volume:", int(segmed_df["has_volume"].sum()))

trainable_df = segmed_df[segmed_df["has_volume"]].reset_index(drop=True)   # <- usable volume+report set
print("trainable_df rows:", len(trainable_df))
trainable_df.head(3)

In [ ]:
old_ct_reports = pd.read_parquet("/cache/fast_data_nas8/vlm_team_data/janhavi/19_June_unique_ct_reports.parquet")

In [ ]:
len(old_ct_reports)

In [ ]:
old_ct_reports.head()

---
# Section 5 — Overlap with `old_ct_reports`

`old_ct_reports` (columns `series_uid`, `study_id`, `report`) is compared against the reports already in
rnd-data-lake. The lake stores reports **content-addressed by text**, so the correct match key is the exact
**report text**. The cell reports how many of your old reports are already in the lake, and how many *extra*
reports the lake has beyond them. (An exact-text match can be thrown off by whitespace differences, so a
whitespace-normalized cross-check is printed too.)

In [ ]:
# Match on exact report TEXT (reports in the lake are content-addressed by text).
lake_reports = set(con.execute("SELECT text FROM nodes_report").fetchdf()["text"])
old_reports  = set(old_ct_reports["report"].dropna())

both      = old_reports & lake_reports
only_old  = old_reports - lake_reports          # in your parquet, missing from the lake
only_lake = lake_reports - old_reports          # extra reports the lake has

print(f"old_ct_reports : {len(old_ct_reports):>9,} rows, {len(old_reports):>9,} unique report texts")
print(f"rnd-data-lake  : {len(lake_reports):>9,} unique report texts (nodes_report)")
print()
print(f"in BOTH                        : {len(both):>9,}  ({100*len(both)/max(len(old_reports),1):.1f}% of old)")
print(f"in old_ct_reports but NOT lake : {len(only_old):>9,}")
print(f"EXTRA in lake (beyond old)     : {len(only_lake):>9,}")

# Whitespace-normalized cross-check (catches near-misses from formatting differences).
import re
norm = lambda s: re.sub(r"\s+", " ", str(s)).strip()
lake_n, old_n = {norm(t) for t in lake_reports}, {norm(t) for t in old_reports}
print(f"\n(normalized) in BOTH: {len(old_n & lake_n):,}   |   extra in lake: {len(lake_n - old_n):,}")

---
# Section 6 — Segmed CXR images (PNG, by **SOP Instance UID**)

Unlike CT (volumetric `.safetensors` keyed by `SeriesInstanceUID`), segmed **CXR** images are single 2D
radiographs saved as **PNGs, one per image**, addressed by **`SOPInstanceUID`**:

```
s3://rnd-data-lake/png/<SOPInstanceUID>.png
```

Resolve them exactly like CT volumes — through `get_cached_path`, never by touching `/mnt/cache` or S3 directly.

**How to get the SOP UIDs** (the join has two wrinkles):
- CXR is modality **`CR`** or **`DX`** — filter `nodes_series` on `"CommonMetadata.Modality" IN ('CR','DX')`.
- The per-image `SOPInstanceUID` lives on **`nodes_instance`** (the series row only has `NumInstances`).
- On `nodes_instance`, `StudyInstanceUID` is **NULL** — so join instances to the segmed CXR **series** on
  `SeriesInstanceUID`, and read study/patient off the series row, not the instance.

> `nodes_instance` is the 123M-row table, so this join is the expensive one in the notebook (tens of seconds).
> Filter to the segmed CXR series set *first* (as below) rather than scanning all instances.
>
> **Note:** the `png/` prefix is being populated — until an image is uploaded, `get_cached_path` raises
> `S3NotFound`, so the fetch cell handles that.


In [2]:
### 6a. Query segmed CXR instances -> per-image PNG URIs (keyed by SOPInstanceUID)
segmed_cxr = con.execute("""
WITH seg AS (                                          -- segmed studies, via the from_source edge
    SELECT substr(e.source_id, 7) AS StudyInstanceUID  -- strip 'study_' prefix
    FROM edges e
    JOIN nodes_source so ON so.node_id = e.target_id
    WHERE e.source_type = 'study' AND e.edge_type = 'from_source' AND so.name = 'segmed'
),
seg_cxr_series AS (                                    -- segmed CR/DX series = CXR (study/patient live here)
    SELECT s.SeriesInstanceUID,
           s.StudyInstanceUID,
           s.PatientID,
           s."CommonMetadata.Modality" AS modality
    FROM nodes_series s
    JOIN seg USING (StudyInstanceUID)
    WHERE s."CommonMetadata.Modality" IN ('CR', 'DX')
)
SELECT
    i.SOPInstanceUID,                                  -- per-image id == the PNG key
    c.SeriesInstanceUID,
    c.StudyInstanceUID,                                -- from the series (instance.StudyInstanceUID is NULL)
    c.PatientID,
    c.modality,
    's3://rnd-data-lake/png/' || i.SOPInstanceUID || '.png' AS image_uri
FROM nodes_instance i
JOIN seg_cxr_series c ON c.SeriesInstanceUID = i.SeriesInstanceUID   -- join on series, NOT study
""").fetchdf()

print("segmed CXR instances:", len(segmed_cxr))
segmed_cxr.head()

segmed CXR instances: 247606


,SOPInstanceUID,SeriesInstanceUID,StudyInstanceUID,PatientID,modality,image_uri
0,1.3.6.1.4.1.55648.1984055079367776049031701063...,1.3.6.1.4.1.55648.1984055079367776049031701063...,1.3.6.1.4.1.55648.1984055079367776049031701063...,Segmed_Patient_5580149431411752902,CR,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.19840...
1,1.3.6.1.4.1.55648.2004823173305962304662291206...,1.3.6.1.4.1.55648.2004823173305962304662291206...,1.3.6.1.4.1.55648.2004823173305962304662291206...,Segmed_Patient_3986238868254577628,CR,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.20048...
2,1.3.6.1.4.1.55648.2016411464896014226714920530...,1.3.6.1.4.1.55648.2016411464896014226714920530...,1.3.6.1.4.1.55648.2016411464896014226714920530...,Segmed_Patient_2445178634411869950,CR,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.20164...
3,1.3.6.1.4.1.55648.2018016261337022994439159504...,1.3.6.1.4.1.55648.2018016261337022994439159504...,1.3.6.1.4.1.55648.2018016261337022994439159504...,Segmed_Patient_7911476742739465529,CR,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.20180...
4,1.3.6.1.4.1.55648.2035220065001680135879051717...,1.3.6.1.4.1.55648.2035220065001680135879051717...,1.3.6.1.4.1.55648.2035220065001680135879051717...,Segmed_Patient_1264749516315474233,CR,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.20352...


In [3]:
old_segmed_data = pd.read_csv("/cache/fast_data_nas71/shankar.ram/data/outputs/12_May_Segmed_data_with_vlm_training_flag_with_view_classes.csv")
len(old_segmed_data)

945039

In [ ]:
old_segmed_data.head()

In [8]:
### 6d. Drop segmed images already used for VLM training (via old_segmed_data)
# WHICH COLUMN TO MERGE ON:
#   - image_filename  -> NO. It's a segmed-internal name "<study>.NNNN.NNNN.green.png"; never matches the lake.
#   - PatientID / StudyInstanceUID / SeriesInstanceUID -> overlap the lake but are too coarse
#     (used_for_vlm_training is per-image, so a study/patient can mix used & unused images).
#   - SOPInstanceUID  -> YES. Exact per-image key; direct string match to the lake's SOPInstanceUID.
# (Only ~35k of the lake's 247k CXR images appear in old_segmed_data at all; the rest use a different
#  UID scheme / aren't in the CSV -> treated as "not in csv" and kept.)
assert {"SOPInstanceUID", "used_for_vlm_training"}.issubset(old_segmed_data.columns)

# robust bool (CSV stores it as the strings "True"/"False")
used_bool = old_segmed_data["used_for_vlm_training"].astype(str).str.strip().str.lower().isin(["true", "1", "yes"])
used_sops = set(old_segmed_data.loc[used_bool, "SOPInstanceUID"].dropna())          # already trained on -> drop
csv_sops  = set(old_segmed_data["SOPInstanceUID"].dropna())                         # present in CSV at all

in_csv  = segmed_cxr["SOPInstanceUID"].isin(csv_sops)
is_used = segmed_cxr["SOPInstanceUID"].isin(used_sops)

# KEEP = not already used  ==  (not in csv)  OR  (used_for_vlm_training == False)
segmed_cxr_new = segmed_cxr[~is_used].reset_index(drop=True)

print(f"segmed_cxr images                     : {len(segmed_cxr):,}")
print(f"  found in old_segmed_data (by SOP)   : {int(in_csv.sum()):,}")
print(f"  already used_for_vlm_training=True   : {int(is_used.sum()):,}  -> dropped")
print(f"  kept (not in csv OR not yet used)   : {len(segmed_cxr_new):,}")
# NOTE: re-run 6a WITHOUT the `LIMIT 20` first, or segmed_cxr here is only the 20-row sample.
segmed_cxr_new.head(3)

segmed_cxr images                     : 247,606
  found in old_segmed_data (by SOP)   : 35,380
  already used_for_vlm_training=True   : 8,313  -> dropped
  kept (not in csv OR not yet used)   : 239,293


,SOPInstanceUID,SeriesInstanceUID,StudyInstanceUID,PatientID,modality,image_uri
0,1.3.6.1.4.1.55648.1984055079367776049031701063...,1.3.6.1.4.1.55648.1984055079367776049031701063...,1.3.6.1.4.1.55648.1984055079367776049031701063...,Segmed_Patient_5580149431411752902,CR,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.19840...
1,1.3.6.1.4.1.55648.2004823173305962304662291206...,1.3.6.1.4.1.55648.2004823173305962304662291206...,1.3.6.1.4.1.55648.2004823173305962304662291206...,Segmed_Patient_3986238868254577628,CR,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.20048...
2,1.3.6.1.4.1.55648.2016411464896014226714920530...,1.3.6.1.4.1.55648.2016411464896014226714920530...,1.3.6.1.4.1.55648.2016411464896014226714920530...,Segmed_Patient_2445178634411869950,CR,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.20164...


In [ ]:
### 6b. Resolve a SOP Instance UID to a local PNG and display it
from PIL import Image
from rnd_data_lake.cache import get_cached_path
from rnd_data_lake.cache.errors import S3NotFound
import matplotlib.pyplot as plt

def load_cxr_png(sop_instance_uid):
    """SOP Instance UID -> local PNG path via the content cache (downloads on first call)."""
    return get_cached_path(f"s3://rnd-data-lake/png/{sop_instance_uid}.png")

sop = segmed_cxr.iloc[0]["SOPInstanceUID"]
try:
    png_path = load_cxr_png(sop)
    img = Image.open(png_path)
    print("resolved:", png_path, "| size:", img.size, "| mode:", img.mode)
    plt.figure(figsize=(5, 5)); plt.imshow(img, cmap="gray"); plt.axis("off")
    plt.title(f"segmed CXR — SOP {sop[:28]}…"); plt.show()
except S3NotFound:
    print(f"No PNG in the bucket yet for SOPInstanceUID={sop}.\n"
          "Segmed CXR PNGs are keyed by SOPInstanceUID at s3://rnd-data-lake/png/<SOPInstanceUID>.png; "
          "once they're uploaded this cell renders the image.")

### 6c. Attach the report to each CXR image

Reports hang off the **study** (`report --report_of--> study`), not the image. A CXR image's series carries
`StudyInstanceUID`, so join image → study → report exactly like the CT report join in **1e/1f**. Reports are
per-study, so every CXR image in a study shares its study's report (PA + lateral of the same visit → same text).

Coverage is effectively total: **247,604 / 247,606** segmed CXR images have a report. Use an inner join to keep
only image+report pairs (drop the last `JOIN` line's `rep` requirement — i.e. `LEFT JOIN` — if you want to see the
handful without one).


In [5]:
### 6c. segmed CXR images + their study report (image_uri, report) pairs
cxr_reports = con.execute("""
WITH seg AS (                                          -- segmed studies, via from_source edge
    SELECT substr(e.source_id, 7) AS StudyInstanceUID
    FROM edges e
    JOIN nodes_source so ON so.node_id = e.target_id
    WHERE e.source_type = 'study' AND e.edge_type = 'from_source' AND so.name = 'segmed'
),
seg_cxr_series AS (                                    -- segmed CR/DX series (study lives here)
    SELECT s.SeriesInstanceUID, s.StudyInstanceUID, s.PatientID
    FROM nodes_series s
    JOIN seg USING (StudyInstanceUID)
    WHERE s."CommonMetadata.Modality" IN ('CR', 'DX')
),
rep AS (                                               -- report text, keyed by study
    SELECT substr(e.target_id, 7) AS StudyInstanceUID, -- strip 'study_' prefix
           r.text AS report
    FROM edges e
    JOIN nodes_report r ON r.node_id = e.source_id
    WHERE e.source_type = 'report' AND e.edge_type = 'report_of'
)
SELECT
    i.SOPInstanceUID,
    c.StudyInstanceUID,
    c.PatientID,
    's3://rnd-data-lake/png/' || i.SOPInstanceUID || '.png' AS image_uri,
    rep.report
FROM nodes_instance i
JOIN seg_cxr_series c ON c.SeriesInstanceUID = i.SeriesInstanceUID   -- image -> series
JOIN rep              ON rep.StudyInstanceUID = c.StudyInstanceUID   -- series' study -> report (INNER = keep only paired)
""").fetchdf()

print("image+report pairs:", len(cxr_reports))
# print(cxr_reports[["SOPInstanceUID", "image_uri"]].to_string())
print("\n--- example report ---\n", cxr_reports.iloc[0]["report"][:800])

image+report pairs: 247604

--- example report ---
 RADIOLOGIC EXAMINATION, RIBS, BILATERAL; 3 VIEWS,RIB XRAY BILATERAL 3 VIEWS

HISTORY: Back pain

Views:  3 oblique.

There are no displaced rib fractures bilaterally.

The ribs reveal no destructive lesions.

See chest radiograph report of same day.

There are no pleural effusions. Costophrenic sulci are sharp.

There is a 2.7 cm ovoid calcification right upper quadrant suggestive of a
gallstone.

IMPRESSION: 

No displaced right rib fractures are identified.

Signed by: segmed_NAME
 Signed Date: segmed_DATE_TIME




In [6]:
cxr_reports.head()

,SOPInstanceUID,StudyInstanceUID,PatientID,image_uri,report
0,1.3.6.1.4.1.55648.3256542949262131222605573477...,1.3.6.1.4.1.55648.3256542949262131222605573477...,Segmed_Patient_4495409089792068913,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.32565...,"RADIOLOGIC EXAMINATION, RIBS, BILATERAL; 3 VIE..."
1,1.3.6.1.4.1.55648.3275280975348592750477276851...,1.3.6.1.4.1.55648.3275280975348592750477276851...,Segmed_Patient_5958128168479105139,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.32752...,XR Chest 2 Views\nPROCEDURE INFORMATION:\nExam...
2,1.3.6.1.4.1.55648.3280360306150747264456960948...,1.3.6.1.4.1.55648.3280360306150747264456960948...,Segmed_Patient_2966760228766106675,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.32803...,"RADIOLOGIC EXAMINATION, CHEST, 2 VIEWS, PA and..."
3,1.3.6.1.4.1.55648.3288961856820510213816464460...,1.3.6.1.4.1.55648.3288961856820510213816464460...,Segmed_Patient_4248415420207275218,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.32889...,"CHEST 2 VWS\nPatient Name: segmed_LASTNAME, se..."
4,1.3.6.1.4.1.55648.3289705985870148502482581596...,1.3.6.1.4.1.55648.3289705985870148502482581596...,Segmed_Patient_5564753713958405065,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.32897...,CHEST XRAY PA AND LATERAL\n\nHistory: Cough.\n...


---
# Section 7 — Paired CT ↔ CXR data (all sources)

"Paired" = the **same patient** has both a **CT** and a **chest X-ray** (`CR`/`DX`). Both `PatientID` and
`Modality` sit on `nodes_series`, so this needs no `edges` join and no source filter (all sources counted).

Grain matters:
- **Patient-level** is the real pairing — a patient's CT and CXR are almost always *separate* studies.
- **Study-level** (CT + CXR in one study) is essentially nonexistent (a DICOM study is single-modality), so don't
  pair on `StudyInstanceUID`.

> Caveat: patient identity here is `PatientID` (the patient node key, `patient_<PatientID>`). If two sources ever
> reused the same `PatientID` string they'd already be merged into one node upstream, which could slightly inflate
> "both". For a training set you'd usually still scope to one source (e.g. segmed) so IDs are unambiguous.


In [ ]:
### 7. How many patients have BOTH a CT and a CXR (all sources)?
pairing = con.execute("""
WITH pat AS (
    SELECT PatientID,
           BOOL_OR("CommonMetadata.Modality" = 'CT')           AS has_ct,
           BOOL_OR("CommonMetadata.Modality" IN ('CR', 'DX'))  AS has_cxr
    FROM nodes_series
    WHERE PatientID IS NOT NULL
    GROUP BY PatientID
)
SELECT COUNT(*)                                    AS patients_total,
       COUNT(*) FILTER (WHERE has_ct)              AS patients_with_ct,
       COUNT(*) FILTER (WHERE has_cxr)             AS patients_with_cxr,
       COUNT(*) FILTER (WHERE has_ct AND has_cxr)  AS patients_with_both   -- <- paired CT+CXR
FROM pat
""").fetchdf()
print(pairing.to_string(index=False))

# The actual paired patient IDs (materialize the cohort). Drop LIMIT to get all ~76k.
paired_patients = con.execute("""
WITH pat AS (
    SELECT PatientID,
           BOOL_OR("CommonMetadata.Modality" = 'CT')           AS has_ct,
           BOOL_OR("CommonMetadata.Modality" IN ('CR', 'DX'))  AS has_cxr
    FROM nodes_series
    WHERE PatientID IS NOT NULL
    GROUP BY PatientID
)
SELECT PatientID FROM pat WHERE has_ct AND has_cxr
""").fetchdf()
# print("\nexample paired patients:")
# print(paired_patients.to_string(index=False))

# Sanity check: CT + CXR in the *same study* is essentially nonexistent (studies are single-modality).
same_study = con.execute("""
WITH st AS (
    SELECT StudyInstanceUID,
           BOOL_OR("CommonMetadata.Modality" = 'CT')           AS has_ct,
           BOOL_OR("CommonMetadata.Modality" IN ('CR', 'DX'))  AS has_cxr
    FROM nodes_series WHERE StudyInstanceUID IS NOT NULL
    GROUP BY StudyInstanceUID
)
SELECT COUNT(*) FILTER (WHERE has_ct AND has_cxr) AS studies_with_ct_and_cxr FROM st
""").fetchone()[0]
# print(f"\nstudies containing BOTH CT and CXR: {same_study}  (confirms patient-level is the right grain)")

In [ ]:
### 7b. How many CXR *studies* are in the paired subset? (vs lake-wide total)
# Paired subset: CXR studies belonging to patients who ALSO have a CT.
paired_cxr = con.execute("""
WITH pat AS (
    SELECT PatientID,
           BOOL_OR("CommonMetadata.Modality" = 'CT')           AS has_ct,
           BOOL_OR("CommonMetadata.Modality" IN ('CR', 'DX'))  AS has_cxr
    FROM nodes_series WHERE PatientID IS NOT NULL
    GROUP BY PatientID
),
paired AS (SELECT PatientID FROM pat WHERE has_ct AND has_cxr)
SELECT COUNT(DISTINCT s.StudyInstanceUID) AS cxr_studies,
       COUNT(*)                           AS cxr_series,
       COUNT(DISTINCT s.PatientID)        AS patients
FROM nodes_series s
JOIN paired USING (PatientID)
WHERE s."CommonMetadata.Modality" IN ('CR', 'DX')
""").fetchdf()

# Lake-wide CXR totals (all patients, paired or not) for comparison.
total_cxr = con.execute("""
SELECT COUNT(*)                         AS cxr_series,
       COUNT(DISTINCT StudyInstanceUID) AS cxr_studies,
       COUNT(DISTINCT PatientID)        AS patients
FROM nodes_series
WHERE "CommonMetadata.Modality" IN ('CR', 'DX')
""").fetchdf()

print("paired subset (patients with BOTH CT & CXR):")
print(paired_cxr.to_string(index=False))
print("\nlake-wide CXR total (all sources, NOT just paired):")
print(total_cxr.to_string(index=False))
print(f"\npaired CXR studies are {100*paired_cxr['cxr_studies'][0]/total_cxr['cxr_studies'][0]:.1f}% "
      "of all CXR studies in the lake")

In [ ]:
### 7c. Of the PAIRED CXR studies, how many reports REFER TO a CT?
# Restricted to CXR studies whose patient ALSO has a CT (the paired subset from 7/7b).
# Keyword proxies over report text (word-boundary CT terms so 'correct'/'structure' don't match).
# Buckets overlap (a report can both compare AND recommend); this is a lexical signal, NOT NLP:
# 'recommends' = recommendation language; 'compares' = reference to an existing/prior CT.
CT   = r'(\b(CT|CTA|HRCT|NCCT|CECT|CTPA)\b|computed tomograph)'
RECO = r'(recommend|advis|suggest|(further|additional) (evaluat|assess|imag|workup)|dedicated|obtain|warrant|if clinically)'
COMP = r'(comparison|correlat|prior|previous|seen on|redemonstrat|since|compared)'

ct_refs = con.execute(f"""
WITH pat_ct AS (                                       -- patients who have a CT
    SELECT DISTINCT PatientID FROM nodes_series WHERE "CommonMetadata.Modality" = 'CT'
),
cxr AS (                                               -- PAIRED CXR studies (patient also has a CT)
    SELECT DISTINCT StudyInstanceUID
    FROM nodes_series
    WHERE "CommonMetadata.Modality" IN ('CR','DX')
      AND PatientID IN (SELECT PatientID FROM pat_ct)
),
rep AS (
    SELECT substr(e.target_id, 7) AS StudyInstanceUID, r.text AS report
    FROM edges e JOIN nodes_report r ON r.node_id = e.source_id
    WHERE e.source_type = 'report' AND e.edge_type = 'report_of'
),
j AS (
    SELECT regexp_matches(rep.report, '{CT}',   'i') AS ct,
           regexp_matches(rep.report, '{RECO}', 'i') AS reco,
           regexp_matches(rep.report, '{COMP}', 'i') AS comp
    FROM cxr JOIN rep USING (StudyInstanceUID)
)
SELECT COUNT(*)                                            AS paired_cxr_studies_with_report,
       COUNT(*) FILTER (WHERE ct)                          AS mentions_ct_any,
       COUNT(*) FILTER (WHERE ct AND reco)                 AS recommends_ct,
       COUNT(*) FILTER (WHERE ct AND comp)                 AS compares_prior_ct,
       COUNT(*) FILTER (WHERE ct AND NOT reco AND NOT comp) AS ct_other_mention
FROM j
""").fetchdf()

n = ct_refs["paired_cxr_studies_with_report"][0]
print(f"paired CXR studies with a report: {n:,}\n")
for col in ["mentions_ct_any", "recommends_ct", "compares_prior_ct", "ct_other_mention"]:
    print(f"{col:22s} {ct_refs[col][0]:>8,}  ({100*ct_refs[col][0]/n:.1f}%)")

In [ ]:
### 7d. Paired CXR reports that MENTION a CT -> (StudyInstanceUID, report, ct_reference_type)
# STUDY-LEVEL only (no images) -> NO nodes_instance join, so this runs in seconds, not minutes.
# ct_reference_type: 'recommended' (advise/suggest a CT), 'prior' (comparison/correlation with an
# existing CT), 'both', or 'unspecified' (CT named but no cue word). Keyword proxy, not NLP.
CT   = r'(\b(CT|CTA|HRCT|NCCT|CECT|CTPA)\b|computed tomograph)'
RECO = r'(recommend|advis|suggest|(further|additional) (evaluat|assess|imag|workup)|dedicated|obtain|warrant|if clinically)'
COMP = r'(comparison|correlat|prior|previous|seen on|redemonstrat|since|compared)'

ct_mention_reports = con.execute(f"""
WITH pat_ct AS (                                       -- patients who have a CT
    SELECT DISTINCT PatientID FROM nodes_series WHERE "CommonMetadata.Modality" = 'CT'
),
cxr_studies AS (                                       -- PAIRED CXR studies (patient also has a CT)
    SELECT DISTINCT StudyInstanceUID
    FROM nodes_series
    WHERE "CommonMetadata.Modality" IN ('CR', 'DX')
      AND PatientID IN (SELECT PatientID FROM pat_ct)
),
rep AS (                                               -- reports that MENTION a CT, keyed by study
    SELECT substr(e.target_id, 7) AS StudyInstanceUID, r.text AS report
    FROM edges e JOIN nodes_report r ON r.node_id = e.source_id
    WHERE e.source_type = 'report' AND e.edge_type = 'report_of'
      AND regexp_matches(r.text, '{CT}', 'i')
)
SELECT DISTINCT
    c.StudyInstanceUID,
    rep.report,
    CASE
        WHEN regexp_matches(rep.report, '{RECO}', 'i') AND regexp_matches(rep.report, '{COMP}', 'i') THEN 'both'
        WHEN regexp_matches(rep.report, '{RECO}', 'i')                                                THEN 'recommended'
        WHEN regexp_matches(rep.report, '{COMP}', 'i')                                                THEN 'prior'
        ELSE 'unspecified'
    END AS ct_reference_type
FROM cxr_studies c
JOIN rep ON rep.StudyInstanceUID = c.StudyInstanceUID
""").fetchdf()

print("paired CXR studies whose report mentions a CT:", len(ct_mention_reports))
print("unique report texts:", ct_mention_reports["report"].nunique())
print(ct_mention_reports["ct_reference_type"].value_counts().to_string())
ct_mention_reports.head()

In [ ]:
len(ct_mention_reports)

In [ ]:
ct_mention_reports.to_parquet("/cache/fast_data_nas71/janhavi/data/ct_training_data/16_july_paired_ct_cxr_reports_filter.parquet")

---
# Section 8 — Combined CXR set: old-segmed test frontals + fresh data-lake images

One dataframe (`combined_df`) with **`image_path`**, **`SOPInstanceUID`**, **`report`** (+ a `source` flag), from two parts:

1. **old_segmed_data** — testing images only: `used_for_vlm_training == False` **and** `is_frontal == True`.
   Image path = `"/raid3/segmed_data/segmed_images/{image_filename}"`; report from the CSV's `report` column.
2. **data-lake `segmed_cxr`** — images **not** used for VLM training or **not** in the CSV (that's `segmed_cxr_new`
   from **6d**). Image path = the S3 URI (`s3://rnd-data-lake/png/<SOPInstanceUID>.png`); report pulled from the
   lake via `cxr_reports` (**6c**), joined on `SOPInstanceUID`.

> Requires **6a** (run without `LIMIT`), **6c** (`cxr_reports`), and **6d** (`segmed_cxr_new`) to have run first.


In [9]:
### Section 8 — build combined_df (image_path, SOPInstanceUID, report)
def _as_bool(s):
    """Robust truthiness for bool or 'True'/'False'-string columns."""
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])

# 1) old_segmed_data — TESTING frontals: used_for_vlm_training == False AND is_frontal == True
old_test = old_segmed_data[(~_as_bool(old_segmed_data["used_for_vlm_training"]))
                           & _as_bool(old_segmed_data["is_frontal"])].copy()
df_old = pd.DataFrame({
    "image_path":     "/raid3/segmed_data/segmed_images/" + old_test["image_filename"].astype(str),
    "SOPInstanceUID": old_test["SOPInstanceUID"].astype(str),
    "report":         old_test["report"],
    "source":         "old_segmed_test",
})

# 2) data-lake segmed_cxr not yet used for training (segmed_cxr_new from 6d),
#    with the lake report attached via SOPInstanceUID (cxr_reports from 6c).
lake_rep  = cxr_reports[["SOPInstanceUID", "report"]].drop_duplicates("SOPInstanceUID")
lake_keep = segmed_cxr_new.merge(lake_rep, on="SOPInstanceUID", how="left")
df_lake = pd.DataFrame({
    "image_path":     lake_keep["image_uri"],                 # s3://rnd-data-lake/png/<SOPInstanceUID>.png
    "SOPInstanceUID": lake_keep["SOPInstanceUID"].astype(str),
    "report":         lake_keep["report"],
    "source":         "data_lake_segmed",
})

# De-duplicate on SOPInstanceUID, PREFERRING the data-lake (S3) row when an image is in both.
# df_lake is concatenated first, so keep="first" keeps the S3 row on a tie.
n_before = len(df_old) + len(df_lake)
combined_df = (pd.concat([df_lake, df_old], ignore_index=True)
               .drop_duplicates("SOPInstanceUID", keep="first")
               .reset_index(drop=True))

print(f"old_segmed testing frontals : {len(df_old):,}")
print(f"data-lake unused images     : {len(df_lake):,}  (missing report: {int(df_lake['report'].isna().sum()):,})")
print(f"combined (pre-dedup)        : {n_before:,}")
print(f"combined_df (deduped by SOP): {len(combined_df):,}  (removed {n_before - len(combined_df):,} dup SOPs)")
print(combined_df["source"].value_counts().to_string())
combined_df.head()

old_segmed testing frontals : 162,912
data-lake unused images     : 239,293  (missing report: 2)
combined (pre-dedup)        : 402,205
combined_df (deduped by SOP): 397,840  (removed 4,365 dup SOPs)
source
data_lake_segmed    239293
old_segmed_test     158547


,image_path,SOPInstanceUID,report,source
0,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.19840...,1.3.6.1.4.1.55648.1984055079367776049031701063...,"XRAY CHEST XRAY, 1 VIEW, PORTABLE\r\nL/R: \r\...",data_lake_segmed
1,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.20048...,1.3.6.1.4.1.55648.2004823173305962304662291206...,CHEST XRAY PA AND LATERAL\n\nHISTORY: R05.3 Ch...,data_lake_segmed
2,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.20164...,1.3.6.1.4.1.55648.2016411464896014226714920530...,"RADIOLOGIC EXAMINATION, CHEST, 2 VIEWS, PA and...",data_lake_segmed
3,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.20180...,1.3.6.1.4.1.55648.2018016261337022994439159504...,"RADIOLOGIC EXAMINATION, CHEST, 2 VIEWS, PA and...",data_lake_segmed
4,s3://rnd-data-lake/png/1.3.6.1.4.1.55648.20352...,1.3.6.1.4.1.55648.2035220065001680135879051717...,"CHEST 2 VWS\nPatient Name: segmed_LASTNAME, se...",data_lake_segmed


In [10]:
len(combined_df)

397840

In [11]:
combined_df.source.value_counts()

source
data_lake_segmed    239293
old_segmed_test     158547
Name: count, dtype: int64

In [12]:
combined_df.to_parquet("/cache/fast_data_nas71/janhavi/data/vlm_eval/16_july_2026_old_segmed_and_data_lake_eval_400k.parquet")